# Pipeline, evaluation và error analysis

## Mục tiêu
- Đặt preprocessing và model trong cùng pipeline.
- Tune sau khi có baseline; giữ test set cuối cùng.
- Phân tích lỗi thay vì chỉ báo một metric.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_wine
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

wine = load_wine(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(wine.data, wine.target, stratify=wine.target, random_state=42)
numeric_features = X_train.columns.tolist()

In [ ]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])
pipeline = Pipeline([
    ('prepare', ColumnTransformer([('numeric', numeric_pipeline, numeric_features)])),
    ('model', LogisticRegression(max_iter=2_000)),
])
pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)
print(classification_report(y_test, predictions, target_names=wine.target_names))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, predictions, display_labels=wine.target_names)

# Xem vài false predictions với feature gốc để tìm pattern.
errors = X_test.assign(actual=y_test, predicted=predictions).query('actual != predicted')
errors.head()

## Những lỗi phổ biến
- Leakage: feature chỉ có sau prediction time.
- Tuning trên test set đến khi điểm đẹp.
- Bỏ qua class imbalance và cost of error.
- Feature importance bị hiểu như causal effect.

## Tự luyện
1. Thay Logistic Regression bằng RandomForestClassifier và so sánh bằng cross-validation.
2. Thêm missing values nhân tạo vào một cột để kiểm tra imputer.
3. Viết ba giả thuyết cho các dòng bị dự đoán sai.

**Checkpoint:** Pipeline được fit trên train và dùng nguyên vẹn để predict test/production.